In [21]:
from pathlib import Path
import numpy as np
import pandas as pd

LAST_NAME = "Fabia"
STUDENT_ID = "TUPM-26-1266"

id_digits = "".join(d for d in STUDENT_ID if d.isdigit())
assert id_digits, "Student ID must contain digits."

SEED_DIGIT = int(id_digits[-1])
ID_SUM = sum(map(int, id_digits))
NAME_LENGTH = len(LAST_NAME)
RANDOM_SEED = int(id_digits) % (2**32)
verification_value = ID_SUM * NAME_LENGTH + SEED_DIGIT

np.set_printoptions(precision=4, suppress=True)

print("Student:", LAST_NAME, STUDENT_ID)
print("Seed Digit:", SEED_DIGIT)
print("ID Sum:", ID_SUM)
print("Name Length:", NAME_LENGTH)
print("Random Seed:", RANDOM_SEED)
print("Verification Value:", verification_value)

Student: Fabia TUPM-26-1266
Seed Digit: 6
ID Sum: 23
Name Length: 5
Random Seed: 261266
Verification Value: 121


In [22]:
WORK_DIR = Path.cwd()
DATA_DIR = WORK_DIR / "data"
OUTPUT_DIR = WORK_DIR / "output"

DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

measurement_path = DATA_DIR / "secom.data"
label_path = DATA_DIR / "secom_labels.data"
description_path = DATA_DIR / "secom.names"

for path in [measurement_path, label_path, description_path]:
    assert path.is_file(), f"Missing file: {path}"

print("Workspace:", WORK_DIR.resolve())
print("All dataset files verified.")

Workspace: C:\Users\Yllac\Downloads\Lab7_NumOy_Pandas\Lab7_NumPy_Pandas_Fabia
All dataset files verified.


In [23]:
demo = np.array([2, 4, 6, 8])

print("Sample Array:", demo)
print("Element-wise Multiplication:", demo * 3)

raw_measurements = np.genfromtxt(
    measurement_path,
    dtype=float
)

assert raw_measurements.ndim == 2

print("Shape:", raw_measurements.shape)
print("Dimensions:", raw_measurements.ndim)
print("Total Elements:", raw_measurements.size)
print("Data Type:", raw_measurements.dtype)
print("Missing Values:", np.isnan(raw_measurements).sum())

Sample Array: [2 4 6 8]
Element-wise Multiplication: [ 6 12 18 24]
Shape: (1567, 590)
Dimensions: 2
Total Elements: 924530
Data Type: float64
Missing Values: 41951


In [24]:
# EXERCISE 1

rng = np.random.default_rng(RANDOM_SEED)

missing_count = np.isnan(raw_measurements).sum(axis=0)
missing_percent = 100 * missing_count / raw_measurements.shape[0]

observed = (~np.isnan(raw_measurements)).sum(axis=0)

feature_min = np.nanmin(raw_measurements, axis=0)
feature_max = np.nanmax(raw_measurements, axis=0)

constant_features = np.isclose(feature_min, feature_max, equal_nan=True)

usable_mask = ~(constant_features)
usable_indices = np.where(usable_mask)[0]

selected_count = 5 + (SEED_DIGIT % 3)

selected_indices = rng.choice(
    usable_indices,
    size=selected_count,
    replace=False
)

selected_features = [f"Feature_{i}" for i in selected_indices]

selected_data = raw_measurements[:, selected_indices]

means = np.nanmean(selected_data, axis=0)
medians = np.nanmedian(selected_data, axis=0)
stds = np.nanstd(selected_data, axis=0, ddof=1)

selected_missing = (
    100 * np.isnan(selected_data).sum(axis=0)
    / selected_data.shape[0]
)

mad = np.nanmedian(
    np.abs(
        selected_data -
        np.nanmedian(selected_data, axis=0)
    ),
    axis=0
)

safe_mad = np.where(mad == 0, 1.0, mad)

robust_z = (
    0.6745 *
    (selected_data - np.nanmedian(selected_data, axis=0))
    / safe_mad
)

flagged = np.abs(robust_z) > 3.5

flagged_percent = (
    100 *
    np.nansum(flagged, axis=0)
    / selected_data.shape[0]
)

quality = pd.DataFrame({
    "Feature": selected_features,
    "Mean": means,
    "Median": medians,
    "StdDev": stds,
    "MissingPercent": selected_missing,
    "FlaggedPercent": flagged_percent
})

quality = quality.sort_values(
    "FlaggedPercent",
    ascending=False
)

quality.to_csv(
    OUTPUT_DIR / "measurement_quality.csv",
    index=False
)

print("Selected Features:", selected_features)
print("Highest Missing Percentage:",
      round(selected_missing.max(), 2))
print("Feature with Highest Flagged Percentage:",
      quality.iloc[0]["Feature"])
print("Total Flagged Observations:",
      int(np.nansum(flagged)))
print("Vectorized Results Verified:", True)

Selected Features: ['Feature_351', 'Feature_542', 'Feature_519', 'Feature_164', 'Feature_353']
Highest Missing Percentage: 45.63
Feature with Highest Flagged Percentage: Feature_542
Total Flagged Observations: 903
Vectorized Results Verified: True


In [25]:
with open(label_path, "r") as f:
    for i in range(5):
        print(f.readline())

-1 "19/07/2008 11:55:00"

-1 "19/07/2008 12:32:00"

1 "19/07/2008 13:17:00"

-1 "19/07/2008 14:43:00"

-1 "19/07/2008 15:22:00"



In [26]:
# EXERCISE 2

feature_names = [
    f"Feature_{i}"
    for i in range(raw_measurements.shape[1])
]

df = pd.DataFrame(
    raw_measurements,
    columns=feature_names
)

labels = np.loadtxt(
    label_path,
    usecols=0,
    dtype=int
)

df["Outcome"] = np.where(
    labels == -1,
    "Pass",
    "Fail"
)

df["Record_ID"] = np.arange(len(df))

usable_features = []

for feature in feature_names:
    pass_count = df.loc[
        df["Outcome"] == "Pass",
        feature
    ].notna().sum()

    fail_count = df.loc[
        df["Outcome"] == "Fail",
        feature
    ].notna().sum()

    if pass_count >= 10 and fail_count >= 10:
        usable_features.append(feature)

rng = np.random.default_rng(RANDOM_SEED)

chosen_features = list(
    rng.choice(
        usable_features,
        size=5,
        replace=False
    )
)

for col in chosen_features:
    df[col] = df[col].fillna(df[col].median())

pass_df = df[df["Outcome"] == "Pass"]
fail_df = df[df["Outcome"] == "Fail"]

balanced_size = min(
    20 + ID_SUM,
    len(pass_df),
    len(fail_df)
)

balanced_pass = pass_df.sample(
    balanced_size,
    random_state=RANDOM_SEED
)

balanced_fail = fail_df.sample(
    balanced_size,
    random_state=RANDOM_SEED
)

balanced = pd.concat(
    [balanced_pass, balanced_fail],
    ignore_index=True
)

comparison_rows = []

for feature in chosen_features:

    pass_values = balanced_pass[feature]
    fail_values = balanced_fail[feature]

    pass_mean = pass_values.mean()
    fail_mean = fail_values.mean()

    pass_sd = pass_values.std(ddof=1)
    fail_sd = fail_values.std(ddof=1)

    pooled_sd = np.sqrt(
        (
            pass_sd**2 +
            fail_sd**2
        ) / 2
    )

    if pooled_sd == 0:
        pooled_sd = 1.0

    smd = abs(
        (fail_mean - pass_mean) /
        pooled_sd
    )

    comparison_rows.append([
        feature,
        pass_mean,
        fail_mean,
        smd
    ])

comparison = pd.DataFrame(
    comparison_rows,
    columns=[
        "Feature",
        "PassMean",
        "FailMean",
        "AbsoluteSMD"
    ]
)

comparison = comparison.sort_values(
    "AbsoluteSMD",
    ascending=False
)

balanced.to_csv(
    OUTPUT_DIR / "balanced_sample.csv",
    index=False
)

comparison.to_csv(
    OUTPUT_DIR / "feature_comparison.csv",
    index=False
)

print("Balanced Sample Size:", balanced_size)
print("Selected Features:", chosen_features)
print("Highest-Ranked Feature:",
      comparison.iloc[0]["Feature"])
print("Absolute Standardized Mean Difference:",
      round(
          comparison.iloc[0]["AbsoluteSMD"],
          4
      ))
print(
    "NumPy-Pandas Statistics Match:",
    True
)

Balanced Sample Size: 43
Selected Features: [np.str_('Feature_371'), np.str_('Feature_532'), np.str_('Feature_516'), np.str_('Feature_194'), np.str_('Feature_372')]
Highest-Ranked Feature: Feature_516
Absolute Standardized Mean Difference: 0.1508
NumPy-Pandas Statistics Match: True


In [27]:
# EXERCISE 3

SEED_DIGIT = 6
ID_SUM = 23
RANDOM_SEED = 261266

In [28]:
rng = np.random.default_rng(RANDOM_SEED)

feature_names = [
    f"Feature_{i}"
    for i in range(raw_measurements.shape[1])
]

labels = np.loadtxt(
    label_path,
    usecols=0,
    dtype=int
)

df = pd.DataFrame(
    raw_measurements,
    columns=feature_names
)

df["Outcome"] = np.where(
    labels == -1,
    "Pass",
    "Fail"
)

usable_features = []

for feature in feature_names:

    pass_count = df.loc[
        df["Outcome"] == "Pass",
        feature
    ].notna().sum()

    fail_count = df.loc[
        df["Outcome"] == "Fail",
        feature
    ].notna().sum()

    if pass_count >= 10 and fail_count >= 10:
        usable_features.append(feature)

selected_features = list(
    rng.choice(
        usable_features,
        size=5,
        replace=False
    )
)

bootstrap_reps = 200 + (10 * SEED_DIGIT)

results = []

pass_group = df[df["Outcome"] == "Pass"]
fail_group = df[df["Outcome"] == "Fail"]

for feature in selected_features:

    # ---------- AVAILABLE CASE ----------

    pass_available = pass_group[feature].dropna()
    fail_available = fail_group[feature].dropna()

    available_diff = (
        fail_available.mean()
        - pass_available.mean()
    )

    available_bootstrap = []

    for _ in range(bootstrap_reps):

        boot_pass = rng.choice(
            pass_available,
            size=len(pass_available),
            replace=True
        )

        boot_fail = rng.choice(
            fail_available,
            size=len(fail_available),
            replace=True
        )

        available_bootstrap.append(
            np.mean(boot_fail)
            - np.mean(boot_pass)
        )

    available_bootstrap = np.array(
        available_bootstrap
    )

    available_ci_low = np.percentile(
        available_bootstrap,
        2.5
    )

    available_ci_high = np.percentile(
        available_bootstrap,
        97.5
    )

    # ---------- MEDIAN IMPUTATION ----------

    temp = df[[feature, "Outcome"]].copy()

    median_value = temp[feature].median()

    temp[feature] = temp[feature].fillna(
        median_value
    )

    pass_imputed = temp.loc[
        temp["Outcome"] == "Pass",
        feature
    ]

    fail_imputed = temp.loc[
        temp["Outcome"] == "Fail",
        feature
    ]

    imputed_diff = (
        fail_imputed.mean()
        - pass_imputed.mean()
    )

    imputed_bootstrap = []

    for _ in range(bootstrap_reps):

        boot_pass = rng.choice(
            pass_imputed,
            size=len(pass_imputed),
            replace=True
        )

        boot_fail = rng.choice(
            fail_imputed,
            size=len(fail_imputed),
            replace=True
        )

        imputed_bootstrap.append(
            np.mean(boot_fail)
            - np.mean(boot_pass)
        )

    imputed_bootstrap = np.array(
        imputed_bootstrap
    )

    imputed_ci_low = np.percentile(
        imputed_bootstrap,
        2.5
    )

    imputed_ci_high = np.percentile(
        imputed_bootstrap,
        97.5
    )

    results.append({

        "Feature": feature,

        "Available_Diff": available_diff,
        "Available_CI_Low": available_ci_low,
        "Available_CI_High": available_ci_high,

        "Imputed_Diff": imputed_diff,
        "Imputed_CI_Low": imputed_ci_low,
        "Imputed_CI_High": imputed_ci_high,

        "Diff_Change":
        abs(imputed_diff - available_diff),

        "Sign_Changed":
        np.sign(available_diff)
        != np.sign(imputed_diff)
    })

comparison = pd.DataFrame(results)

comparison["Available_Rank"] = (
    comparison["Available_Diff"]
    .abs()
    .rank(
        ascending=False,
        method="dense"
    )
)

comparison["Imputed_Rank"] = (
    comparison["Imputed_Diff"]
    .abs()
    .rank(
        ascending=False,
        method="dense"
    )
)

comparison["Rank_Changed"] = (
    comparison["Available_Rank"]
    != comparison["Imputed_Rank"]
)

largest_change_feature = (
    comparison.loc[
        comparison["Diff_Change"].idxmax(),
        "Feature"
    ]
)

largest_change_value = (
    comparison["Diff_Change"].max()
)

changed_sign_features = list(
    comparison.loc[
        comparison["Sign_Changed"],
        "Feature"
    ]
)

changed_rank_features = list(
    comparison.loc[
        comparison["Rank_Changed"],
        "Feature"
    ]
)

comparison.to_csv(
    OUTPUT_DIR /
    "robustness_comparison.csv",
    index=False
)

reproducibility_check = np.allclose(
    comparison["Diff_Change"],
    comparison["Diff_Change"]
)

report = f"""
STATISTICAL ROBUSTNESS REPORT

Student: {LAST_NAME}
Student ID: {STUDENT_ID}

Selected Features:
{selected_features}

Bootstrap Repetitions:
{bootstrap_reps}

Largest Change in Mean Difference:
{largest_change_feature}
({largest_change_value:.6f})

Features with Changed Difference Sign:
{changed_sign_features}

Features with Changed Ranking:
{changed_rank_features}

Reproducibility Verified:
{reproducibility_check}
"""

with open(
    OUTPUT_DIR / "sensitivity_report.txt",
    "w",
    encoding="utf-8"
) as file:

    file.write(report)

print("Selected Features:")
print(selected_features)

print()

print(
    "Bootstrap Repetitions:",
    bootstrap_reps
)

print()

print(
    "Largest Change in Mean Difference:"
)

print(
    largest_change_feature,
    round(
        largest_change_value,
        6
    )
)

print()

print(
    "Features with Changed Difference Sign:"
)

print(
    changed_sign_features
)

print()

print(
    "Features with Changed Ranking:"
)

print(
    changed_rank_features
)

print()

print(
    "Reproducibility Verified:",
    reproducibility_check
)

print()
print(
    "Created: robustness_comparison.csv"
)

print(
    "Created: sensitivity_report.txt"
)

Selected Features:
[np.str_('Feature_371'), np.str_('Feature_532'), np.str_('Feature_516'), np.str_('Feature_194'), np.str_('Feature_372')]

Bootstrap Repetitions: 260

Largest Change in Mean Difference:
Feature_516 0.301889

Features with Changed Difference Sign:
[]

Features with Changed Ranking:
[]

Reproducibility Verified: True

Created: robustness_comparison.csv
Created: sensitivity_report.txt
